# RaCA vs MRA on the FULL SemanticKITTI (molab, branch `farquleet_add_module`)

Trains one configuration on the **full** dataset (3,834 training frames, 815 validation frames = sequence 08), built by
`kaggle/build_full_dataset.ipynb` and stored as 4 private Kaggle datasets (`semantickitti-mra-full-d/-a/-b/-c`).
Run it twice, once per method, with the same settings: `RUN = "raca_fullkitti"` (RaCA C1+C2+C3+C4) and
`RUN = "mra_fullkitti"` (original MRA-T baseline).

**Before running**
1. Notebook specs button -> turn the **GPU** on. `.env` next to the notebook with `KAGGLE_API_TOKEN=<your token>`.
2. Settings cell: `RUN` and `EPOCHS` (12 by default, ~11 h per run; MRA's paper uses 24, ~22 h).
3. Run all cells top to bottom and **keep this tab open and the computer awake** (molab stops ~90 min after the page is closed).

**Time per session:** data download + checks ~30-45 min (every new session: molab clears `/tmp`), then training.
One epoch = 3,834 steps (~50 min); validation every 2 epochs (~6 min).

**Sessions end after 12 h.** Training stops itself after `TRAIN_HOURS` (10.5 h) and the newest epoch checkpoint, the best
checkpoint and the logs are kept in `<RUN>_resume/` next to this notebook (survives restarts). Next session: open the
notebook and run all cells again - it downloads the data again and **continues from the last finished epoch**.
At the end, `<RUN>_results.zip` (logs, results, best and final model) is created next to the notebook: download it.


In [ ]:
import os
# ---- Settings ----
RUN       = "raca_fullkitti"                           # raca_fullkitti (RaCA C1-C4) or mra_fullkitti (MRA baseline)
REPO_URL  = "https://github.com/fafnirLore/RaCA.git"
BRANCH    = "farquleet_add_module"                     # RaCA code + kaggle/ + molab/ folders
EPOCHS    = 12                                         # same for both methods (MRA paper: 24)
TRAIN_HOURS = "10.5h"                                  # stop before molab's 12 h limit; next session resumes
EXTRA     = f"--cfg-options runner.max_epochs={EPOCHS} total_epochs={EPOCHS}"

# Where the dataset is. One of:
#   "auto"                               search this machine for .../dataset/sequences (notebook folder, home, /data, /mnt, ...)
#   "/path/to/folder" or an archive      a folder above dataset/ (e.g. .../kitti_subset), or a .zip / .tar(.gz) of it
#   "hf://datasets/<user>/<repo>"        a Hugging Face dataset repository (private: put HF_TOKEN in the notebook's .env)
#   "kaggle://datasets/<user>/<slug>"    a Kaggle dataset            } private: put KAGGLE_API_TOKEN=<your token> in the
#   "kaggle://kernels/<user>/<slug>"     a Kaggle notebook's output  } notebook's .env (kaggle.com -> Settings -> API -> Generate New Token)
KAGGLE_USER = "daniyalimran"
DATA      = ",".join(f"kaggle://datasets/{KAGGLE_USER}/semantickitti-mra-full-{g}" for g in "dabc")   # KAGGLE_API_TOKEN in .env

# Where to copy the newest checkpoint, logs and results while training (every 10 minutes), so they survive the session.
#   ""                                   no backup (download the files yourself before the notebook shuts down)
#   "/path/to/folder"                    e.g. a mounted bucket
#   "hf://datasets/<user>/<repo>/runs"   a private Hugging Face dataset repository (HF_TOKEN with write access in .env)
BACKUP_TO = ""

# Google Drive folder that receives logs, results and checkpoints after every epoch ("" = off), e.g. "RaCA_runs".
# Connect Drive first: Files sidebar -> Add remote storage -> Google Drive.
DRIVE_DIR = ""

REPO      = "/tmp/RaCA"                             # code
OUT_ROOT  = "/tmp/mra_runs"                            # checkpoints, logs, results: OUT_ROOT/RUN/
RESUME_DIR = os.path.join(os.getcwd(), f"{RUN}_resume")   # survives molab restarts (files < 1 GB)
print("run:", RUN, "| epochs:", EPOCHS, "| branch:", BRANCH, "| data:", DATA, "| backup:", BACKUP_TO or "off", "| drive:", DRIVE_DIR or "off")

In [ ]:
# Helper: run a shell command and show its output line by line as it is produced.
import os, subprocess

if os.path.exists(".env"):                             # KAGGLE_API_TOKEN / HF_TOKEN, if molab has not loaded them already
    for _line in open(".env"):
        _k, _sep, _v = _line.strip().partition("=")
        if _sep and _k and not _k.startswith("#"):
            os.environ.setdefault(_k.strip(), _v.strip().strip("'\""))

def sh(cmd, on_line=None, **extra_env):
    env = dict(os.environ, RUN=RUN, REPO=REPO, REPO_URL=REPO_URL, BRANCH=BRANCH, EXTRA=EXTRA,
               OUT_ROOT=OUT_ROOT, BACKUP_TO=BACKUP_TO, PYTHONUNBUFFERED="1", **extra_env)
    proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
        if on_line:
            on_line(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {cmd}" + chr(10) + "--- last output lines ---" + chr(10) + "".join(lines[-40:]))
    return "".join(lines)

RUN_DIR = f"{OUT_ROOT}/{RUN}"
MRA_PY  = "/tmp/mra_env/bin/python"

In [ ]:
# Google Drive copy of the run folder (off when DRIVE_DIR is ""). Drive is reached through fsspec from this notebook's
# own Python process, because that is where molab keeps the Drive sign-in; the training subprocess cannot see it.
import glob
_drive_sent = {}                                         # local path -> (size, mtime) already uploaded

def drive_fs():
    from gdrive_fsspec import GoogleDriveFileSystem
    return GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True)

def drive_sync(checkpoints=True):
    """Copy new or changed files of RUN_DIR to <DRIVE_DIR>/<RUN>/. Never raises: a Drive error must not stop training."""
    if not DRIVE_DIR:
        return
    try:
        fs, remote = drive_fs(), f"{DRIVE_DIR}/{RUN}"
        fs.makedirs(remote, exist_ok=True)
        local = [f for f in glob.glob(f"{RUN_DIR}/*") if os.path.isfile(f) and not os.path.islink(f)]
        for f in sorted(local, key=os.path.getsize):     # small files (logs, json) first
            if f.endswith(".pth") and not checkpoints:
                continue
            st = os.stat(f); key = (st.st_size, st.st_mtime)
            if _drive_sent.get(f) != key:
                fs.put(f, f"{remote}/{os.path.basename(f)}"); _drive_sent[f] = key
                print(f"[drive] {os.path.basename(f)} ({st.st_size / 2**20:.1f} MB) -> {remote}/", flush=True)
        keep = {os.path.basename(f) for f in local}
        for r in fs.ls(remote, detail=False):            # checkpoints training has replaced: remove the Drive copy too
            if r.endswith(".pth") and os.path.basename(r) not in keep:
                fs.rm(r); print(f"[drive] removed old {os.path.basename(r)}", flush=True)
    except Exception as e:
        print(f"[drive] SYNC FAILED ({type(e).__name__}: {e}) - training continues; files remain in {RUN_DIR}/", flush=True)

def drive_check():
    """Write, read back and delete a small file, so a Drive problem shows before training and not after an epoch."""
    if not DRIVE_DIR:
        print("Google Drive copy is off (DRIVE_DIR is empty)"); return
    fs, remote = drive_fs(), f"{DRIVE_DIR}/{RUN}"
    fs.makedirs(remote, exist_ok=True)
    with fs.open(f"{remote}/.write_test", "wb") as f:
        f.write(b"ok")
    with fs.open(f"{remote}/.write_test", "rb") as f:
        assert f.read() == b"ok", "Drive write test: content read back differs"
    fs.rm(f"{remote}/.write_test")
    print(f"Google Drive ok: writing to {remote}/")


In [ ]:
# 1. Code + software (5-10 min the first time; skipped when already installed in this session)
step1_log = sh("""
set -e
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || { echo "No GPU visible: turn it on with the notebook specs button, then re-run."; exit 1; }
if [ -d "$REPO/.git" ]; then cd "$REPO" && git fetch -q origin "$BRANCH" && git checkout -q "$BRANCH" && git reset -q --hard "origin/$BRANCH"
else git clone -q -b "$BRANCH" "$REPO_URL" "$REPO" && cd "$REPO"; fi
git log --oneline -1    # must show the commit you expect; an existing clone is reset to the branch head on GitHub
bash molab/install.sh
""")

In [ ]:
# 2. Data: download the 4 full-dataset parts from Kaggle (~30-45 min), link them at /tmp/kitti_full/ and check every file
_ = step1_log
sh('df -h /tmp | tail -1')   # the 4 datasets need ~70-80 GB free while downloading
step2_log = sh(f'cd "$REPO" && {MRA_PY} molab/prepare_data.py "$DATA" /tmp/kitti_full/', DATA=DATA)
DATA_ROOT = step2_log.strip().splitlines()[-1].split("DATA_ROOT=")[1]

In [ ]:
# 3. Starting weights: ImageNet ResNet-50 + SGN-T, combined into ckpts/warm_start.pth (~1 min).
#    Same starting point as the Kaggle run and the RaCA runs. If training was backed up earlier, bring it back first.
_ = DATA_ROOT
step3_log = sh("""
set -e
cd "$REPO" && bash molab/get_weights.sh
if [ -n "$BACKUP_TO" ]; then /tmp/mra_env/bin/python molab/backup.py pull "$OUT_ROOT/$RUN" "$BACKUP_TO"; fi
""")

In [ ]:
# 4. Smoke test (~1 min): two real training steps and one inference step on this GPU, with the peak memory.
#    If this passes, the software, the data and the starting weights work together.
_ = step3_log
step4_log = sh('cd "$REPO" && source molab/env.sh && PYTHONPATH=$REPO $PY molab/smoke_test.py molab/configs/$RUN.py $SMOKE_EXTRA',
               SMOKE_EXTRA=EXTRA if EXTRA.strip().startswith("--cfg-options") else "")

In [ ]:
# 5. Train (stops after TRAIN_HOURS; if it says TRAINING FAILED with exit 124 that is the time limit:
#    next session, run all cells again and it continues). Lines like `Epoch [1][50/3834] ... loss: ...` show progress, `EarlyStoppingHook: ...` lines the monitored value
#    and the patience counter; resumes from the last saved epoch if there is one.
_ = step4_log
if glob.glob(f"{RUN_DIR}/results_trained.json"):
    step5_log = "already trained and evaluated"
    print(step5_log, "- delete", RUN_DIR, "to train again")
else:
    drive_check()
    import shutil
    os.makedirs(RESUME_DIR, exist_ok=True); os.makedirs(RUN_DIR, exist_ok=True)

    def keep_for_resume():
        """Copy the newest epoch checkpoint, the best checkpoint and the logs to RESUME_DIR (atomic, never raises)."""
        try:
            latest = os.path.join(RUN_DIR, "latest.pth")
            if os.path.exists(latest):
                shutil.copy(os.path.realpath(latest), f"{RESUME_DIR}/latest.pth.part")
                os.replace(f"{RESUME_DIR}/latest.pth.part", f"{RESUME_DIR}/latest.pth")
            for b in glob.glob(f"{RUN_DIR}/best_*.pth"):
                if not os.path.exists(f"{RESUME_DIR}/{os.path.basename(b)}"):
                    for old in glob.glob(f"{RESUME_DIR}/best_*.pth"):
                        os.remove(old)
                    shutil.copy(b, f"{RESUME_DIR}/{os.path.basename(b)}.part")
                    os.replace(f"{RESUME_DIR}/{os.path.basename(b)}.part", f"{RESUME_DIR}/{os.path.basename(b)}")
            for f in glob.glob(f"{RUN_DIR}/*.log") + glob.glob(f"{RUN_DIR}/*.json"):
                shutil.copy(f, RESUME_DIR)
            print(f"[resume] saved to {RESUME_DIR}", flush=True)
        except Exception as e:
            print(f"[resume] copy failed ({type(e).__name__}: {e}) - training continues", flush=True)

    # New session: bring back the last finished epoch, the best checkpoint and the logs from RESUME_DIR
    if os.path.exists(f"{RESUME_DIR}/latest.pth") and not os.path.exists(f"{RUN_DIR}/latest.pth"):
        for f in glob.glob(f"{RESUME_DIR}/*"):
            if not f.endswith(".part"):
                shutil.copy(f, RUN_DIR)
        print("[resume] restored", sorted(os.listdir(RUN_DIR)), flush=True)
    _pending = [False]
    def _on_line(line):
        # "Saving checkpoint" is logged before the file is complete, so copy at the next epoch-level line: the validation
        # summary of that epoch, or the first progress line of the next epoch when validation is off.
        if "Saving checkpoint" in line:
            _pending[0] = True
        elif _pending[0] and ("Epoch(val)" in line or "Epoch [" in line):
            _pending[0] = False
            drive_sync()
            keep_for_resume()
    try:
        step5_log = sh('cd "$REPO" && bash molab/train.sh molab/configs/$RUN.py $RUN $EXTRA', on_line=_on_line,
                       TIMEOUT=TRAIN_HOURS)
    finally:
        drive_sync()                                     # last epoch, and whatever exists if training failed
        keep_for_resume()


In [ ]:
# 6. Evaluate on the validation frames (a few minutes): the best checkpoint kept by the evaluation hook, else the last one
_ = step5_log
step6_log = sh("""
set -e
cd "$REPO"
CKPT=$(ls -t "$OUT_ROOT/$RUN"/best_*.pth 2>/dev/null | head -1); CKPT=${CKPT:-$OUT_ROOT/$RUN/latest.pth}
if [ -s "$OUT_ROOT/$RUN/results_trained.json" ]; then echo "results already computed:"; head -12 "$OUT_ROOT/$RUN/results_trained.json"
else echo "evaluating $CKPT"; bash molab/eval.sh molab/configs/$RUN.py "$CKPT" "$OUT_ROOT/$RUN/results_trained.json" "$RUN: trained"; fi
""")


In [ ]:
# 7. Keep the results: final backup (if BACKUP_TO is set), plus a folder and a zip next to this notebook with the logs,
#    results, config and the trained models (best by validation mIoU and last; ~400 MB each, the zip must stay under
#    molab's 1 GB limit). Download the zip from the file browser.
_ = step6_log
import shutil
keep_dir = os.path.join(os.getcwd(), f"{RUN}_results")
os.makedirs(keep_dir, exist_ok=True)
for _f in glob.glob(f"{RUN_DIR}/*.json") + glob.glob(f"{RUN_DIR}/*.log") + glob.glob(f"{RUN_DIR}/*.py"):
    shutil.copy(_f, keep_dir)
_best = sorted(glob.glob(f"{RUN_DIR}/best_*.pth"), key=os.path.getmtime)
if _best:
    shutil.copy(_best[-1], f"{keep_dir}/{RUN}_best.pth")                               # the checkpoint that was evaluated
if os.path.exists(f"{RUN_DIR}/latest.pth"):
    shutil.copy(os.path.realpath(f"{RUN_DIR}/latest.pth"), f"{keep_dir}/{RUN}_final.pth")   # the real file, not the link
shutil.make_archive(keep_dir, "zip", keep_dir, ".")
step7_log = sh("""
if [ -n "$BACKUP_TO" ]; then cd "$REPO" && /tmp/mra_env/bin/python molab/backup.py push "$OUT_ROOT/$RUN" "$BACKUP_TO"; fi
du -sh "$OUT_ROOT/$RUN"; ls -la "$OUT_ROOT/$RUN"
""")
drive_sync(checkpoints=False)                            # results_trained.json
for _f in sorted(os.listdir(keep_dir)):
    print(f"{os.path.getsize(os.path.join(keep_dir, _f)) / 1e6:9.1f} MB  {_f}")
print("download:", keep_dir + ".zip")

_ = sh(f'grep -E "Epoch\\(val\\)" {RUN_DIR}/train_full.log | sed -E "s/.*Epoch\\(val\\) \\[([0-9]+)\\].*SC_IoU: ([0-9.]+), semkitti_SSC_mIoU: ([0-9.]+).*/epoch \\1: IoU \\2 mIoU \\3/" || true')
